# micrograd exercises
The following sections will test the knowledge and revise the core concepts learned!

## Section 1: derivatives

In [124]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos, exp, log

def f(a, b, c):
    return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))

6.336362190988558


In [125]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29

def gradf(a, b, c):
    dfda = -3*(a**2) - 0.5*a**(-0.5)
    dfdb = 3*cos(3*b) + 2.5*b**1.5
    dfdc = c**-2
    return [dfda, dfdb, dfdc] # todo, return [df/da, df/db, df/dc]

# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
    ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
    print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390593273
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027111255
OK for dim 2: expected 0.0625, yours returns 0.0625


In [126]:
# now estimate the gradient numerically without any calculus, using
# the approximation we used in the video.
# you should not call the function df from the last cell

# -----------
def compute_gradient(a=2.0, b=3.0, c=4.0):
    h = 0.00000001

    for n in range(3):
        d1 = f(a, b, c)
        if n == 0:
            a += h
            d2 = f(a, b, c)
            df_da = (d2-d1)/h

        if n == 1:
            b += h
            d2 = f(a, b, c)
            df_db = (d2-d1) / h

        if n == 2:
            c += h
            d2 = f(a, b, c)
            df_dc = (d2-d1) / h

    return [df_da, df_db, df_dc]

numerical_grad = compute_gradient(2, 3, 4)
# -----------


for dim in range(3):
    ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
    print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553380251014
OK for dim 1: expected 10.25699027111255, yours returns 10.256990456980475
OK for dim 2: expected 0.0625, yours returns 0.06249987194451023


In [127]:
# there is an alternative formula that provides a much better numerical
# approximation to the derivative of a function.
# learn about it here: https://en.wikipedia.org/wiki/Symmetric_derivative
# implement it. confirm that for the same step size h this version gives a
# better approximation.

# -----------
def compute_sym_gradient(a=2.0, b=3.0, c=4.0):
    h = 0.00000001

    for n in range(3):
        d1 = f(a, b, c)
        if n == 0:
            rd_a = a + h
            ld_a = a - h
            dr = f(rd_a, b, c)
            dl = f(ld_a, b, c)
            df_da = (dr-dl)/(2*h)

        if n == 1:
            rd_b = b + h
            ld_b = b - h
            dr = f(a, rd_b, c)
            dl = f(a, ld_b, c)
            df_db = (dr-dl)/(2*h)

        if n == 2:
            rd_c = c + h
            ld_c = c - h
            dr = f(a, b, rd_c)
            dl = f(a, b, ld_c)
            df_dc = (dr-dl)/(2*h)

    return [df_da, df_db, df_dc]

numerical_grad2 = compute_sym_gradient()
# -----------

for dim in range(3):
    ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
    print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553291433172
OK for dim 1: expected 10.25699027111255, yours returns 10.256990368162633
OK for dim 2: expected 0.0625, yours returns 0.0624999607623522


## section 2: support for softmax

In [128]:
# Value class starter code, with many functions taken out
from math import exp, log

class Value:

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other): # exactly as in the video
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out
        
  # ------
  # re-implement all the other functions needed for the exercises below
  # your code here

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "Only power of int and float types supported for now!"
        out = Value(self.data**other, (self,), f"**{other}")

        def _backward():
            self.grad += other * self.data**(other-1) * out.grad
        out._backward = _backward
        return out
        
    def exp(self):
        x = self.data
        out = Value(exp(x), (self,), 'exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        x = self.data
        out = Value(log(x), (self,), 'log')

        def _backward():
            self.grad += (1/x) * out.grad
        out._backward = _backward
        return out
    
    def __truediv__(self, other):
        return self * other**-1

    def __rmul__(self, other):
        return self * other
        
    def __radd__(self, other):
        return self + other 

    def __neg__(self): # -self # calls self.__mul__(-1)
        return self * -1
  # ------

    def backward(self): # exactly as in video
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
                visited.add(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [129]:
# without referencing our code/video __too__ much, make this cell work
# you'll have to implement (in some cases re-implemented) a number of functions
# of the Value object, similar to what we've seen in the video.
# instead of the squared error loss this implements the negative log likelihood
# loss, which is very often used in classification.

# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
    counts = [logit.exp() for logit in logits]
    print(counts)
    denominator = sum(counts, Value(0.0))
    print(f"Denominator: {denominator}")
    # denominator = sum(counts)
    out = [c / denominator for c in counts]
    print(out)
    return out

# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log() # dim 3 acts as the label for this input example
loss.backward()
print(loss.data)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
    ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
    print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")


[Value(data=1.0), Value(data=20.085536923187668), Value(data=0.1353352832366127), Value(data=2.718281828459045)]
Denominator: Value(data=23.939154034883327)
[Value(data=0.04177257051535045), Value(data=0.839024507462532), Value(data=0.00565330266221633), Value(data=0.11354961935990122)]
2.1755153626167147
OK for dim 0: expected 0.041772570515350445, yours returns 0.041772570515350445
OK for dim 1: expected 0.8390245074625319, yours returns 0.8390245074625319
OK for dim 2: expected 0.005653302662216329, yours returns 0.005653302662216329
OK for dim 3: expected -0.8864503806400986, yours returns -0.8864503806400986


In [131]:
# verify the gradient using the torch library
# torch should give you the exact same gradient
import torch

# Convert logits to torch tensors
torch_logits = torch.tensor([logit.data for logit in logits], requires_grad=True)

# Apply softmax using torch
torch_probs = torch.softmax(torch_logits, dim=0)

# Calculate loss, corresponding to probs[3] as the label
torch_loss = -torch.log(torch_probs[3])

# Perform backward pass
torch_loss.backward()

# Compare gradients
print("\n--- Torch Gradient Verification ---")
for dim in range(4):
  ok = 'OK' if abs(torch_logits.grad[dim].item() - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, torch returns {torch_logits.grad[dim].item()}")


--- Torch Gradient Verification ---
OK for dim 0: expected 0.041772570515350445, torch returns 0.041772566735744476
OK for dim 1: expected 0.8390245074625319, torch returns 0.8390244841575623
OK for dim 2: expected 0.005653302662216329, torch returns 0.005653302650898695
OK for dim 3: expected -0.8864503806400986, torch returns -0.8864504098892212
